# Climate State Finder

The goal of this notebook is to develop a generalized climate state finder. Based on user selected variable and event definitions, this notebook will search all simulations to find these climate states. The inputs required are:

* Domain, a custom shapefile
* One variable
* GWL

**Expected Runtime**: 

In [ ]:
import climakitae as ck
import geopandas as gpd
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

## 1. Make Selections

Use the ClimateData object to subset our data catalog. Here, we'll make selections and see available options for variables, spatial domain, and GWLs.

In [ ]:
cd = ck.ClimateData(verbosity=-1)

### 1a. Select a temporal resolution
The temporal resolution selection will impact which downscaling methods and variables are available.

In [ ]:
# Show temporal resolution (AKA "table id") options 
# cd.show_table_id_options()

In [ ]:
table_id = "mon" # Options are "mon", "day", "1hr"
cd.table_id(table_id)

### 1b. Select spatial resolution
Depending on your selections above, your spatial resolution options may change. Use `cd.show_grid_label_options()` to see the available spatial resolutions for your variable and dataset selections. A smaller resolution (e.g. 3 km) indicates a finer grid (e.g. high resolution) versus a coarser grid (e.g. 45 km). Note that you'll need to use the CMIP6 grid label wording in the code: `d01` (45 km), `d02` ( 9 km), or `d03` ( 3 km). 

In [ ]:
# Show spatial resolution ("grid label") options 
# cd.show_grid_label_options()

In [ ]:
grid_label = "d03" # 3 km 
cd.grid_label(grid_label)

### 1c. Select a baseline and future GWL
The baseline Global Warming Level (GWL) is the historical warming level used as a reference for comparison. We suggest using 0.8 °C. Available warming levels are 0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, and 4.0 °C, where higher values represent a warmer future climate.

In [ ]:
# Set baseline and future global warming level as a float 
baseline_gwl = 0.8
future_gwl = 2.0

### 1f. Select variable and units 
This notebook is configured to run with one of the variable options: "precipitation", "relative humidity", or "air temperature". 

In [ ]:
variable = "air temperature"

Next, the plain-language variable name is mapped to the appropriate variable id for each dataset. The units are set here as well-- feel free to change them to your desired units (unit options are in the code comments). 

**NOTE**: LOCA2 only provides minimum and maximum values for air temperature and relative humidity. If these variables are desired, the notebook will automatically retrieve **both variables** and average them. Because of the additional processing for these LOCA2 variables, the `LOCA2_var` is a `list` rather than a `str`. 

In [ ]:
if variable == "precipitation": 
    WRF_var = "prec"
    WRF_units = "inches" # options: "mm", "inches"
    LOCA2_var = ["pr"]
    LOCA2_units = "inches" # options: "kg m-2 s-1", "mm", "inches"
elif variable == "relative humidity": 
    WRF_var = "rh"
    WRF_units = "%[0 to 100]" # options: "%[0 to 100]" 
    LOCA2_var = ["hursmax","hursmin"] 
    LOCA2_units = "percent" # options: "percent"  (equivalent to %[0 to 100]) 
elif variable == "air temperature": 
    WRF_var = "t2"
    WRF_units = "degF" # options: "K", "degC", "degF"
    LOCA2_var = ["tasmax","tasmin"] 
    LOCA2_units = "degF" # options: "K", "degC", "degF"

### 1e. Select aggregation method
Select a method to aggregate spatially (across the region of interest) and temporally (across time to a monthly timescale). The temporal aggregation step will only applied if you chose daily or hourly data, since monthly data is by definition already aggregated. 

In [ ]:
# options are mean, median, sum
temporal_aggregation = "mean" 
spatial_aggregation = "mean" 

### 1f. Select climate state duration
How often should the climate state be observed to be considered a 'hit', in units of months. 

In [ ]:
duration = 24 # months 

### 1g. Validate selections 
Ensure the inputs are valid before proceeding to data retrieval 

In [ ]:
# Check GWL 
valid_gwls = [0.8, 1.0, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0]
invalid = [gwl for gwl in [baseline_gwl, future_gwl] if gwl not in valid_gwls]
if invalid:
    raise ValueError(f"Invalid GWL value(s) {invalid}. Must be one of: {valid_gwls}")

# Check variable 
valid_variables = ["precipitation", "relative humidity", "air temperature"]
if variable not in valid_variables:
    raise ValueError(f"Invalid climate_variable '{variable}'. Must be one of: {valid_variables}")

# Check aggregation 
valid_aggs = ["mean","median","sum"]
invalid = [agg for agg in [spatial_aggregation, temporal_aggregation] if agg not in valid_aggs]
if invalid:
    raise ValueError(f"Invalid aggregation value(s) {invalid}. Must be one of: {valid_aggs}")

## 2. Select Spatial Domain
A default shapefile is provided so you can explore the notebook workflow without additional setup. The example uses the Brawley hydrologic area in Imperial County, California, which makes up part of the Colorado River basin within the Salton Sea HUC-8 watershed.

To use your own study area, replace the shapefile path in the code below with the path to your custom shapefile.

In [ ]:
# Region of Interest shapefile
shapefile_filepath = "https://geo.sandag.org/server/rest/directories/downloads/Hydrological_Basins_shapefile.zip"
spatial_domain_name = "Brawley"
filter_field = "haname" # column to filter domain name on 

# Read in data and filter for specific ROI 
shapefile_all = gpd.read_file(shapefile_filepath)
roi = shapefile_all[shapefile_all[filter_field] == spatial_domain_name]

# Generate quick plot
roi.plot();

## 3. Retrieve Data

### 3a. Retrieve WRF data 

In [ ]:
wrf_data = (cd.catalog("cadcat")
    .activity_id("WRF")
    .institution_id("UCLA")
    .table_id(table_id)
    .grid_label(grid_label)
    .variable(WRF_var)
    .processes({
        "warming_level": {
            "warming_levels": [baseline_gwl, future_gwl],
            "add_dummy_time": True,
        },
        "clip": roi,
        "convert_units": WRF_units,
        "metric_calc": {
            "metric": spatial_aggregation, 
            "dim": ["x", "y"], # Aggregation across spatial dimensions 
        },
    })
).get()

### 3b. Retrieve LOCA2 data 

In [ ]:
loca2_data = [(cd.catalog("cadcat")
    .activity_id("LOCA2")
    .institution_id("UCSD")
    .table_id(table_id)
    .grid_label(grid_label)
    .variable(var)
    .processes({
        "warming_level": {
            "warming_levels": [baseline_gwl, future_gwl],
            "add_dummy_time": True,
        },
        "clip": roi,
        "convert_units": LOCA2_units,
        "metric_calc": {
            "metric": spatial_aggregation,
            "dim": ["lat", "lon"],
        },
    })
).get() for var in LOCA2_var]

### 3c. Preprocessing 
Next, we will perform some simple data preprocessing to ensure the WRF and LOCA2 data are in the proper format for the next operations. 

In [ ]:
# Convert Dataset --> DataArray object 
wrf_data = wrf_data[WRF_var]
loca2_data = [loca2_data[i][var] for i,var in zip([0,1],LOCA2_var)]

# LOCA2 preprocessing if two variables were retrieved 
if len(loca2_data) > 1: 
    # For air temperature and relative humidity, average the min and max values 
    loca2_data = (loca2_data[0] + loca2_data[1])/2
else: 
    # Otherwise, only one dataset is retrieved, and it's a singeton list 
    loca2_data = loca2_data[0]

# Assign both datasets the same name 
wrf_data.name = variable
loca2_data.name = variable

### 3d. Perform monthly aggreggation 
For daily or hourly data, perform monthly aggregation using method of choice 

In [ ]:
# create monthly aggegations
if table_id != "mon":
    wrf_data = eval(f"wrf_data.resample(time = '1M').{temporal_aggregation}()")
    loca2_data = eval(f"loca2_data.resample(time = '1M').{temporal_aggregation}()")
    print(f"Data was aggregated monthly using a {temporal_aggregation}")

### 3d. Read data into memory 
Dask will print a progress bar as the data is loaded into memory for each dataset. 

In [ ]:
with ProgressBar():
    wrf_data.load()
    loca2_data.load()

## 4. Calculate anomalies

### 4a. Calculate climatological mean of baseline gwl

In [ ]:
# calculate monthly mean for each simulation for the baseline gwl
loca2_clim_mean = loca2_data.drop_duplicates(dim="warming_level").sel(warming_level = float(baseline_gwl)).groupby("time.month").median()
wrf_clim_mean = wrf_data.drop_duplicates(dim="warming_level").sel(warming_level = float(baseline_gwl)).groupby("time.month").median()

### 4b. Calculate anomaly of future and baseline GWL to climatological mean

In [ ]:
# subtract the monthly climatology from the future gwl data to create an anomaly
loca2_anom = loca2_data.drop_duplicates(dim="warming_level").sel(warming_level = float(future_gwl)).groupby("time.month") - loca2_clim_mean
wrf_anom = wrf_data.drop_duplicates(dim="warming_level").sel(warming_level = float(future_gwl)).groupby("time.month") - wrf_clim_mean

In [ ]:
# calculate historical anomaly
hist_loca2_anom = loca2_data.drop_duplicates(dim="warming_level").sel(warming_level = float(baseline_gwl)).groupby("time.month") - loca2_clim_mean
hist_wrf_anom = wrf_data.drop_duplicates(dim="warming_level").sel(warming_level = float(baseline_gwl)).groupby("time.month") - wrf_clim_mean

### 4c. Calculate seasonal rolling average anomaly for baseline and future GWL 

In [ ]:
loca2_anom = loca2_anom.rolling(time=duration).median()
wrf_anom = wrf_anom.rolling(time=duration).median()

In [ ]:
hist_loca2_anom = hist_loca2_anom.rolling(time=duration).median()
hist_wrf_anom = hist_wrf_anom.rolling(time=duration).median()

## 5. Compute climate states 
We will define climate states using quantiles to get the upper (75th) and lower (25th) quantiles of the anomalies 

In [ ]:
# Create an upper and lower threshold 
loca2_high = hist_loca2_anom.quantile(q=0.75,dim="time")
loca2_low = hist_loca2_anom.quantile(q=0.25,dim="time")
wrf_high = hist_wrf_anom.quantile(q=0.75,dim="time")
wrf_low = hist_wrf_anom.quantile(q=0.25,dim="time")

In [ ]:
# Add data to data array
wrf_anom = wrf_anom.assign_coords(climate_state_high = ("sim", wrf_high.values))
wrf_anom = wrf_anom.assign_coords(climate_state_low = ("sim", wrf_low.values))
loca2_anom = loca2_anom.assign_coords(climate_state_high = ("sim", loca2_high.values))
loca2_anom = loca2_anom.assign_coords(climate_state_low = ("sim", loca2_low.values))

Next we will search the data for the climate states defined above

In [ ]:
def compute_climate_hits(anom, high_thresh, low_thresh, duration):
    """
    Identify climate state 'hits' for each simulation and time step.

    Parameters
    ----------
    anom : xr.DataArray
        Anomaly data with dimensions (sim, time).
    high_thresh : xr.DataArray
        Upper threshold per simulation.
    low_thresh : xr.DataArray
        Lower threshold per simulation.
    duration : int
        Number of months required to constitute a hit.

    Returns
    -------
    np.ndarray
        Array of shape (sim, time) with 1 (high), -1 (low), or 0 (no hit).
    """
    climate_hit = np.zeros(anom.values.shape)
    n_times = len(anom["time"])

    for t in range(n_times):
        # get time indices for this window, clipped to array bounds
        time_idx = [i for i in range(t, t + duration) if i < n_times]
        anom_window = anom.isel(time=time_idx)

        # count how many time steps exceed each threshold per simulation
        high_hit = ((anom_window > high_thresh).sum(dim="time") >= duration).values
        low_hit = ((anom_window < low_thresh).sum(dim="time") >= duration).values

        # mark hits: 1 for high state, -1 for low state
        climate_hit[np.ix_(high_hit, time_idx)] = 1
        climate_hit[np.ix_(low_hit, time_idx)] = -1

    return climate_hit

wrf_climate_hit = compute_climate_hits(wrf_anom, wrf_high, wrf_low, duration)
loca2_climate_hit = compute_climate_hits(loca2_anom, loca2_high, loca2_low, duration)

In [ ]:
wrf_anom = wrf_anom.assign_coords(climate_state_hit=(("sim", "time"), wrf_climate_hit))
loca2_anom = loca2_anom.assign_coords(climate_state_hit=(("sim", "time"), loca2_climate_hit))

## Prep for Plotting

In [ ]:
# create a 'model' variable
lsims = loca2_anom.sim.values.tolist()
loca2_models = [s.split("_")[1] for s in lsims]
loca2_anom = loca2_anom.assign_coords(models = ("sim",loca2_models))

In [ ]:
wsims = wrf_anom.sim.values.tolist()
wrf_models = [s.split("_")[1] for s in wsims]
wrf_anom = wrf_anom.assign_coords(models = ("sim",wrf_models))

In [ ]:
# create a realization variable
loca2_realization = [s.split("_")[2]+'_'+s.split("_")[3] for s in lsims]
loca2_anom = loca2_anom.assign_coords(realization = ("sim",loca2_realization))
wrf_realization = [s.split("_")[2]+'_'+s.split("_")[3] for s in wsims]
wrf_anom = wrf_anom.assign_coords(realization = ("sim",wrf_realization))

In [ ]:
# create a data frame to make plotting easier
loca2_df = loca2_anom.to_dataframe().reset_index()
wrf_df = wrf_anom.to_dataframe().reset_index()

In [ ]:
# Create a downscaling name
loca2_df["downscaling"] = "loca2"
wrf_df["downscaling"] = "wrf"

In [ ]:
# combine into one 
final_df = pd.concat([loca2_df, wrf_df], ignore_index=True, keys=loca2_df.keys())

In [ ]:
final_df.head()

## Plot Yay!

In [ ]:
# plot all the data 
def annotate(data, **kws):
    ax = plt.gca()
    ax.axhline(0, ls='--',color="black")
    ax.fill_between(
        data.time,data[variable],
        where=data.climate_state_hit == 1,
        facecolor='maroon',
        alpha=.5
    )
    ax.fill_between(
        data.time,
        data[variable],
        where=data.climate_state_hit == -1,facecolor='darkblue',alpha=.5)

g1 = sns.relplot(
    data=final_df,
    x="time", y=variable,
    col="sim",color="black",col_wrap=4,
    kind="line", 
    height=5, aspect=1, facet_kws=dict(sharex=False), 
)
g1.map_dataframe(annotate)
plt.show()

In [ ]:
# Save the figure to a file
g1.savefig(f'climate_state_finder_{variable}_under_{future_gwl}gwl_with_{duration}{table_id}duration.jpeg'.replace(" ", "_"))

In [ ]:
# save climate state data (to be loaded into event finder)
final_df.to_parquet(f'climate_state_{variable}_under_{future_gwl}gwl_with_{duration}{table_id}duration'.replace(" ", "_"))